<a href="https://colab.research.google.com/github/GiovannaFarani/ProjetoLACEDA/blob/main/Desafio_LACEDA_Giovanna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalação e importação

In [1]:
!pip install pyspark thefuzz

In [2]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, substring
from thefuzz import process

In [3]:
pd.set_option('display.max_columns', None)

In [4]:
base_menor = '/content/2026_teste_tecnico_relinke_cidacs(in).csv'
base_maior = '/content/29_BA.csv'

In [5]:
df_menor = pd.read_csv(base_menor, sep=';')

### PARTE 1 - Seleção das linhas necessárias

Selecionar apenas os municípios da base menor que precisam ser procurados na base maior

In [6]:
municipios = df_menor['consulta_municipio'].dropna().astype(str).str.upper().unique().tolist()
municipios

['290070',
 '290130',
 '290200',
 '290240',
 '290320',
 '290630',
 '290660',
 '290700',
 '290750',
 '291072',
 '291080',
 '291160',
 '291170',
 '291180',
 '291210',
 '291430',
 '291480',
 '291560',
 '291600',
 '291680',
 '291770',
 '291840',
 '291880',
 '291910',
 '291955',
 '292145',
 '292390',
 '292440',
 '292520',
 '292530',
 '292580',
 '292740',
 '292880',
 '292895',
 '293050',
 '293077',
 '293135',
 '293200',
 '293220',
 '293250',
 '293320',
 '293360',
 '290135',
 '290260',
 '290323',
 '290340',
 '290390',
 '290650',
 '290720',
 '290730',
 '290840',
 '290940',
 '290960',
 '291100',
 '291190',
 '291390',
 '291460',
 '291570',
 '291610',
 '291690',
 '291750',
 '291800',
 '292220',
 '292240',
 '292265',
 '292300',
 '292380',
 '292480',
 '292500',
 '292710',
 '292720',
 '292790',
 '292870',
 '292910',
 '292925',
 '292950',
 '293015',
 '293170']

Incia o spark e carrega a base maior

In [7]:
spark = SparkSession.builder \
.appName("Record_linkage_LACEDA") \
.config("spark.driver.memory", "4g") \
.getOrCreate()

In [8]:
spark = SparkSession.builder.appName("Record_Linkage_LACEDA").getOrCreate()
df_base_spark = spark.read.csv(base_maior, header=True, sep=';')

In [9]:
df_base_spark.head()

Row(COD_UNICO_ENDERECO='112026678', COD_UF='29', COD_MUNICIPIO='2900801', COD_DISTRITO='290080125', COD_SUBDISTRITO='29008012500', COD_SETOR='290080125000007P', NUM_QUADRA='0', NUM_FACE='126', CEP='45910000', DSC_LOCALIDADE='POUSO ALEGRE', NOM_TIPO_SEGLOGR='BAIXA', NOM_TITULO_SEGLOGR=None, NOM_SEGLOGR='SEM DENOMINACAO', NUM_ENDERECO='0', DSC_MODIFICADOR='SN', NOM_COMP_ELEM1=None, VAL_COMP_ELEM1=None, NOM_COMP_ELEM2=None, VAL_COMP_ELEM2=None, NOM_COMP_ELEM3=None, VAL_COMP_ELEM3=None, NOM_COMP_ELEM4=None, VAL_COMP_ELEM4=None, NOM_COMP_ELEM5=None, VAL_COMP_ELEM5=None, LATITUDE='-17.421301', LONGITUDE='-39.588591', NV_GEO_COORD='1', COD_ESPECIE='3', DSC_ESTABELECIMENTO=None, COD_INDICADOR_ESTAB_ENDERECO='1', COD_INDICADOR_CONST_ENDERECO=None, COD_INDICADOR_FINALIDADE_CONST=None, COD_TIPO_ESPECI=None)

In [10]:
df_base_spark.limit(5).toPandas()

,COD_UNICO_ENDERECO,COD_UF,COD_MUNICIPIO,COD_DISTRITO,COD_SUBDISTRITO,COD_SETOR,NUM_QUADRA,NUM_FACE,CEP,DSC_LOCALIDADE,NOM_TIPO_SEGLOGR,NOM_TITULO_SEGLOGR,NOM_SEGLOGR,NUM_ENDERECO,DSC_MODIFICADOR,NOM_COMP_ELEM1,VAL_COMP_ELEM1,NOM_COMP_ELEM2,VAL_COMP_ELEM2,NOM_COMP_ELEM3,VAL_COMP_ELEM3,NOM_COMP_ELEM4,VAL_COMP_ELEM4,NOM_COMP_ELEM5,VAL_COMP_ELEM5,LATITUDE,LONGITUDE,NV_GEO_COORD,COD_ESPECIE,DSC_ESTABELECIMENTO,COD_INDICADOR_ESTAB_ENDERECO,COD_INDICADOR_CONST_ENDERECO,COD_INDICADOR_FINALIDADE_CONST,COD_TIPO_ESPECI
0,112026678,29,2900801,290080125,29008012500,290080125000007P,0,126,45910000,POUSO ALEGRE,BAIXA,None,SEM DENOMINACAO,0,SN,None,None,None,None,None,None,None,None,None,None,-17.421301,-39.588591,1,3,None,1,None,None,None
1,112026678,29,2900801,290080125,29008012500,290080125000007P,0,126,45910000,POUSO ALEGRE,BAIXA,None,SEM DENOMINACAO,0,SN,None,None,None,None,None,None,None,None,None,None,-17.421301,-39.588591,1,1,None,None,None,None,101
2,192615207,29,2901007,290100705,29010070500,290100705000002P,1,2,45300000,AMARGOSA,AVENIDA,DOUTOR,ABELARDO VELOSO,776,None,APARTAMENTO,1ANDAR AP103,None,None,None,None,None,None,None,None,-13.030280,-39.595209,1,1,None,None,None,None,103
3,24076305,29,2901007,290100705,29010070500,290100705000029P,0,5,45300000,LAGOA QUEIMADA,POVOADO,None,LAGOA QUEIMADA,17,None,CASA,None,None,None,None,None,None,None,None,None,-12.976211,-39.577448,1,1,None,None,None,None,101
4,192618976,29,2901007,290100705,29010070500,290100705000045P,4,15,45300000,SANTA RITA,TRAVESSA,None,QUINTA LUIS SANDE,0,SN,TERREO,None,None,None,None,None,None,None,None,None,-13.016349,-39.608956,1,6,DEPOSITO,1,None,None,None


Filtra da base maior as linha com os municipios que pertecem a base menor (procurada)

* obs: foi utilizado o substring para não cosiderar o último dígito, pois segundo o IBGE é apenas um dígito verificador

In [11]:
df_base_filtrada = df_base_spark.filter(substring(col('COD_MUNICIPIO'), 1, 6).isin(municipios))

In [12]:
df_base_final = df_base_filtrada.select('COD_MUNICIPIO', 'NOM_SEGLOGR', 'COD_SETOR')

Volta a base para o padrão do pandas

In [13]:
df_base_final = df_base_final.toPandas()

In [15]:
df_base_final.head()

,COD_MUNICIPIO,NOM_SEGLOGR,COD_SETOR
0,2900702,SEVERINO CALDEIRA,290070220000001P
1,2900702,BURY,290070205000186P
2,2900702,JOSE JOAQUIM LEAL,290070205000070P
3,2900702,JOSE JOAQUIM LEAL,290070205000070P
4,2901353,ISAURA,290135305000008P


Finaliza a execução do spark

In [16]:
spark.stop()

### PARTE 2 - LIMPEZA

Cria as colunas de logradouro limpas. Elas serão utilizadas para definir a paridade no algoritmo.

In [17]:
df_menor['endereco_limpo'] = df_menor['consulta_logradouro'].astype(str).str.upper().str.replace(r'[^\w\s]', '', regex=True).str.strip()
df_base_final['endereco_limpo'] = df_base_final['NOM_SEGLOGR'].astype(str).str.upper().str.replace(r'[^\w\s]', '', regex=True).str.strip()

Criação da coluna com os seis digitos reais do código do município.

In [20]:
df_base_final['codigo_municipio'] = df_base_final['COD_MUNICIPIO'].astype(str).str[:6]

PARTE 3 - MATCHING

In [21]:
setores_match = []

In [22]:
#visita linha a linha da base menor
for index, linha in df_menor.iterrows():
    mun_alvo = str(linha['consulta_municipio']).split('.')[0][:6]
    end_alvo = linha['endereco_limpo']

    cidades = df_base_final[df_base_final['codigo_municipio'] == mun_alvo]

    if not cidades.empty:
        lista_ruas = cidades['endereco_limpo'].tolist()

        # utiliza o fuzzy match para atirbuir nota de áridade para cada linha
        best_match = process.extractOne(end_alvo, lista_ruas)

        # se a nota >= 85, considera um acerto
        if best_match and best_match[1] >= 85:
            rua_encontrada = best_match[0]
            # busca o codigo do setor que correspondente a essa rua
            setor = cidades[cidades['endereco_limpo'] == rua_encontrada]['COD_SETOR'].iloc[0]
            setores_match.append(setor)
        else:
            setores_match.append("Não localizado")
    else:
        setores_match.append("Cidade sem dados")


### 4 - Resultado Final

In [24]:
df_menor['SETOR_CENSITARIO_FINAL'] = setores_match
saida = '/content/resultado_relink.csv'
df_menor.to_csv(saida, sep=';', index=False)

print(f"Cruzamento concluído com sucesso! Ficheiro gerado em: {saida}")

Cruzamento concluído com sucesso! Ficheiro gerado em: /content/resultado_relink.csv
